# Panel Data

Sometimes, data comes in such a way that many observations share certain common features. With financial data, this is commonly seen as many security prices tracked across the same time period. We treat this kind of data differently to time series data, as we will explore in this notebook.

## Loading

As always, let's start by importing pandas and loading and cleaning our dataset.

In [ ]:
import pandas as pd

# Load the data
df = pd.read_csv("https://raw.githubusercontent.com/ImperialCollegeLondon/efds-ta-python/refs/heads/main/pyfi-data/SP500_2025.csv")

# Convert the 'datadate' column to a datetime object
df.datadate = pd.to_datetime(df.datadate)

# Look at the dataframe
df

We'll stop short of setting the index as our datetime value though. This is because an index shines when we have unique values, and because this panel data contains lots of different company stocks for just one quarter of a year, we'll see the same date lots of times.

In [ ]:
# From the dataframe above we know there are nearly 127,000 rows

# How many unique dates exist in the data frame
print("Number of unique dates", df.datadate.nunique())

# Last date in the dataset
print("Last date", df.datadate.max())

# First date in the dataset
print("First date", df.datadate.min())

# Number of unique companies vs number of unique tickers
print("Unique companies", df["conm"].nunique())
print("Unique tickers:", df["tic"].nunique())
# Notice the discrepancy between these values - we'll look more at why this is when we learn to group

## Cleaning

Let's not forget data cleaning! Are the dates ordered? Do we have missing data?

### Sorting

In [ ]:
# Visually inspecting dates shows they are out of order

# We previously used sort_index - but dates aren't in the index here!
# Instead we use sort_values, which can take multiple column names
# We sort first by ticker, then by date, which shows 
df.sort_values(["tic", "datadate"], inplace=True)

# Visually inspect the sort - notice the index numbers are off
display(df)
display(df.iloc[0])
display(df.loc[0])

# We can reset it, while dropping the old numbers
df.reset_index(drop=True, inplace=True)

# Alternatively - pass ignore_index=True in sort_values()

# Visually inspect
df

### Missing/Duplicated Values


In [ ]:
# Check for missing values
df.info()

# Check for duplicates
print("Duplicates:", df.duplicated().sum())

# Display missing data
display(df[df.isnull().any(axis=1)])

# Some open price data is missing, on days where the
# trading volume is zero. Because of this, we can fairly
# safely drop these rows (think of them as another weekend!)
df = df.dropna()

# Dropping rows puts gaps in our index - we can reset it if we want
df.reset_index(drop=True, inplace=True)

# Check missing data again...
print("Missing data after cleaning", df.isnull().sum().sum())
df

## Exploring

Let's explore this panel data a bit more, to answer some questions:

- Which exchanges are considered
- Which exchanges appear most


In [ ]:
# If we use unique() instead of nunique() we'll get the actual values
print("Unique exhanges:", df["exchg"].unique())

# value_counts will give us appearances by number
print("Exchanges by appearance:", df["exchg"].value_counts())

## Grouping

What if we wanted to calculate daily returns in this data set. Is it as simple as using `pct_change()`? Let's try.

In [ ]:
df["returns"] = df["prccd"].pct_change()
df.iloc[248:252]

Can you see what's gone wrong here? Our first calculated daily return for Apple is using Agilent's last closing price. This hopefully gets across the importance of *grouping*, particularly useful with this kind of panel data.


We can solve this with the `groupby()` method of data frames.

In [ ]:
df["Returns"] = df.groupby("tic")["prccd"].pct_change()
df.iloc[248:252]

Perfect! Grouping is a very powerful way to manipulate panel data. Once you've grouped, you can call functions and they will be applied groupwise as we saw above. Here are some other common functions with groups:

In [ ]:
# Identify the number of rows in each group
print("Number of rows per group", df.groupby("tic").size())

# Subset a specific group
apple = df.groupby("tic").get_group("AAPL")
apple.set_index("datadate", inplace=True)
apple # Time series data once again!

Let's see what else we can do with grouping. Recall that we had more tickers than companies. Let's see why that is by looking at how many unique tickers belong to each company (using `tic` and `conm`). Then let's list those companies.

In [ ]:
# First we create a series with the number of unique stocks for each company
ticker_counts = df.groupby("conm")["tic"].nunique()
multi_tic = ticker_counts[ticker_counts > 1]
multi_tic

### Exercise: Tick Tick

Identify the number of unique tickers traded on each exchange.

In [ ]:
## YOUR CODE GOES HERE

## Aggregation

Aggregation functions like `mean()`, `median()`, `sum()`, `min()`, `max()`, `first()`, `last()` and `std()` can be applied to grouped data to give insights across panel data. Say we wanted the average daily return of each traded security, or the max volume traded on any given day for each security?

The exercises above helped us identify that the `tic` column corresponds to unique tickers, so let's use that for grouping from now on. 

In [ ]:
df.groupby("tic")["returns"].mean()

Useful, but only to a point. If a ticker symbol is unfamiliar to us, the company name is useful information. What if we want both ticker and company name for the security? Let's look at grouping by multiple columns to help!

In [ ]:
# Simply pass a list - as_index is optional and leaves our columns as columns
display(df.groupby(["conm", "tic"], as_index=False)["prccd"].min()) 

# Notice first() is different to min() - it is the first price of the period
display(df.groupby(["conm", "tic"], as_index=False)["prccd"].first())

# A common use of first() is for aggregating like, non-numeric data
df.groupby(["conm", "tic"], as_index=False)["conm"].first()

# Note as_index is *nearly* the same as calling reset_index(), though
# reset_index() will fail when a indexed column is also accessed as in the above
# df.groupby(["conm", "tic"])["conm"].first().reset_index() # fails

Once we've done these sorts of aggregation, we're often curious to see who sits at the top or the bottom of the distribution. We can use `nlargest()` and its antonym here. Note that `as_index=False` doesn't work here (since `nlargest` and `nsmallest` require the index) but `reset_index()` does work (since it applies after those functions).

In [ ]:
# The lowest average return
display(df.groupby(["conm", "tic"])["returns"].mean().nsmallest().reset_index())

# The largest maximum daily return
df.groupby(["conm", "tic"])["returns"].max().nlargest().reset_index()

We can also group by multiple columns! This can be helpful when doing aggregation, for example, to find high performers in each month. Because our date is just a regular column, we need to specify `.dt` to use any datetime functions.

In [ ]:
# First create a column to specify the month
df["month"] = df.datadate.dt.month_name()

# Then use it to group and aggregate for max closing price each month
df.groupby(["tic", "month"], as_index=False).prccd.max()

### Exercise: Trading Exchanges

Identify the total trading volume of each exchange.

In [ ]:
## YOUR CODE GOES HERE

### Exercise: Big Days

Identify the three days of the week that see the highest total trading volume in the data (and show the total trading volume across those days). 

In [ ]:
## YOUR CODE GOES HERE

### Exercise: The 1000 Club

For securities that reached a closing price above 1000, how many times in each month, did they acheive this?

In [ ]:
## YOUR CODE GOES HERE

## Multiple Aggregation

We can use the `agg()` method, and pass it a dictionary to do multiple aggregations at once on grouped data. This can be helpful for further analyses, or for producing a more descriptive aggregated data frame.

In [ ]:
display(df.groupby("tic").agg({"conm": "first", "prccd": "mean"}))

(
    df.groupby("tic")
    .agg(
        {
            "conm": "first",
            "prccd": ["first", "last"],
            "cshtrd": "sum",
        }
    )
    .nlargest(5, columns=("cshtrd", "sum"))
    .set_axis(
        ["conm", "first close", "last close", "total volume"],
        axis=1,
    )
)

### Exercise: Quick Quarter Query

Using multiple aggregation, create an aggregated data frame with ticker and company name, the first open price in the period for each ticker and the last close price in the period for each ticker. Create a new column in this aggregated data frame that shows the price difference between the initial open and the final close for each ticker.

In [ ]:
## YOUR CODE GOES HERE